# clikernel MCP server

Serves this instance's Python session over HTTP so a laptop harness (Claude Code, codex) can run code here, and shows the bearer token behind a Solveit sign-in restricted to one email.

Two ports, both needing a port mapping in the Solveit instance settings:

- `6001` -> the MCP endpoint (`/mcp`), guarded by the token
- `8003` -> this token page, guarded by Google sign-in

Named `zz_` so it sorts last: `solveit.core.autorun` runs `AUTORUN/*.ipynb` in sorted order and a failure here would stop later notebooks from starting.

In [ ]:
import json, os, re, secrets, socket, subprocess, time
from pathlib import Path

NAME = 'virgil-client-portal'   # the name the laptop registers this instance under
OWNER_EMAIL = 'eg@answer.ai'    # the only email allowed to see the token
MCP_PORT, PAGE_PORT = 6001, 8003
LOG = Path('/app/data/.clikernel_mcp.log')

tok_path = Path('/app/data/.clikernel_token')
if not tok_path.exists():
    tok_path.write_text(secrets.token_urlsafe(32))
    tok_path.chmod(0o600)
TOKEN = tok_path.read_text().strip()

def listening(port, host='127.0.0.1'):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex((host, port)) == 0

listening(MCP_PORT), listening(PAGE_PORT)

(False, False)

In [ ]:
if not listening(MCP_PORT):
    # PYTHONSAFEPATH and cwd='/': a `clikernel` directory in the worker's cwd would shadow the installed package
    env = dict(os.environ, CLIKERNEL_TOKEN=TOKEN, PYTHONSAFEPATH='1')
    log = LOG.open('a')
    mcp_proc = subprocess.Popen(
        ['clikernel-mcp', '--transport', 'streamable-http', '--host', '0.0.0.0', '--port', str(MCP_PORT)],
        env=env, cwd='/', stdout=log, stderr=log)
    time.sleep(2)
listening(MCP_PORT)

True

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi
from dialoghelper.solve_auth import setup_solve_signin, solve_signin_rt, sub_from_signin, SolveSigninError

MCP_URL = f'https://{json.loads(os.environ["PUBLIC_DOMAINS"])[str(MCP_PORT)]}/mcp'
ADD_CMD = f'claude mcp add --transport http {NAME} {MCP_URL} -H "Authorization: Bearer {TOKEN}"'

app, rt = fast_app(pico=False, live=False)
setup_solve_signin(app, port=PAGE_PORT, email_re=re.escape(OWNER_EMAIL))

@rt('/')
def get(session):
    if 'auth' not in session:
        return Titled('clikernel MCP', P(f'Sign in as {OWNER_EMAIL} to see this instance\'s token.'),
                      A('Sign in with Solveit', href=solve_signin_rt))
    return Titled('clikernel MCP', H3('Register this instance'), Pre(Code(ADD_CMD)),
                  H3('Token'), Pre(Code(TOKEN)), P(f'Endpoint: {MCP_URL}'), A('Sign out', href='/sign_out'))

@rt
def signin_completed(session, signin_reply:str=''):
    try: session['auth'] = sub_from_signin(session, signin_reply)
    except SolveSigninError as e: return Titled('Sign-in refused', P(str(e)))
    return RedirectResponse('/', status_code=303)

@rt
def sign_out(session):
    session.pop('auth', '')
    return RedirectResponse('/', status_code=303)

server = JupyUvi(app, port=PAGE_PORT)

## Notes

- Rotate the token: delete `/app/data/.clikernel_token`, re-run this notebook, re-register on the laptop.
- Server logs: `/app/data/.clikernel_mcp.log`.
- The MCP server only starts if nothing already holds port 6001, so a hand-started server is left alone. That one has its own token, which is not the token shown on the page.
- Dialogs under `AUTORUN` are exempt from the idle reaper, so this keeps running.